In [ ]:
import requests
import time
from notebookutils import mssparkutils
from datetime import datetime, timezone
from pyspark.sql.functions import max as spark_max
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType

execution_start_time = datetime.now(timezone.utc)

# --- Configuration ---
kusto_cluster_uri = "https://trd-6uegjpfbf030eemxtw.z1.kusto.fabric.microsoft.com"
kusto_database = "MonitoringEventhouse"

tenant_id = "your-tenant-id"
client_id = "your-client-id"
client_secret = "your-client-secret"
# client_secret = mssparkutils.credentials.getSecret("your-kv", "your-secret")

fabric_api_base = "https://api.fabric.microsoft.com/v1"
fix_aop_config_table = "Fix_AOP_Config"


In [ ]:
# Create Config Table in Lakehouse (if not exists)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {fix_aop_config_table} (
        LastRunTimestamp TIMESTAMP,
        UpdatedAt TIMESTAMP
    )
    USING DELTA
""")

print(f"Verified configuration table: {fix_aop_config_table}")


In [ ]:
def get_spn_token(scope):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': scope
    }
    try:
        response = requests.post(url, data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except Exception as e:
        print(f"Failed to retrieve token for scope {scope}: {e}")
        return None

fabric_token = get_spn_token("https://api.fabric.microsoft.com/.default")
kusto_token  = get_spn_token(f"{kusto_cluster_uri}/.default")
print("Tokens retrieved." if fabric_token and kusto_token else "ERROR: Failed to retrieve tokens.")

In [ ]:
# --- Read watermark from Fix_AOP_Config Delta table ---
watermark_ts = None

df_config = spark.table(fix_aop_config_table)
max_row = df_config.agg(spark_max("LastRunTimestamp").alias("max_ts")).collect()
max_ts = max_row[0]["max_ts"]

if max_ts:
    watermark_ts = max_ts
    if watermark_ts.tzinfo is None:
        watermark_ts = watermark_ts.replace(tzinfo=timezone.utc)
    print(f"Watermark loaded: {watermark_ts}")
else:
    print("No watermark in Fix_AOP_Config — will process all records.")


In [ ]:
# Build Kusto query, filtered by watermark when available
upper_bound = execution_start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
watermark_filter = (
    f'| where ingestion_time() > datetime("{watermark_ts.strftime("%Y-%m-%dT%H:%M:%SZ")}") and ingestion_time() <= datetime("{upper_bound}")'
    if watermark_ts else
    f'| where ingestion_time() <= datetime("{upper_bound}")'
)

kusto_query = f"""
AOPAlertLogs
{watermark_filter}
| summarize arg_max(ingestion_time(), *) by WorkspaceId
| where AlertStatus in ("EmailSent", "NoEmail")
| where ToChangeAOP == true
| project WorkspaceId, AOP, AOPMustBe
"""

try:
    df = spark.read \
        .format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("kustoCluster", kusto_cluster_uri) \
        .option("kustoDatabase", kusto_database) \
        .option("kustoQuery", kusto_query) \
        .option("accessToken", kusto_token) \
        .load()

    workspaces = df.collect()
    print(f"Found {len(workspaces)} workspace(s) to fix.")
except Exception as e:
    print(f"Error reading from Kusto: {e}")
    workspaces = []


In [ ]:
def get_workspace_comm_policy(workspace_id):
    url = f"{fabric_api_base}/workspaces/{workspace_id}/networking/communicationPolicy"
    headers = {"Authorization": f"Bearer {fabric_token}"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json(), response.headers.get("ETag")


def set_workspace_aop(workspace_id, aop_must_be):
    """
    Applies AOP policy to a workspace.
    Returns (status, detail, error_code):
      - Success: ("Success", outbound_action, None)
      - Failure: ("Failed", error_description, error_code_str)
    """
    outbound_action = "Deny" if aop_must_be == "EnableWorkspaceOutboundAccessProtection" else "Allow"

    try:
        current_policy, etag = get_workspace_comm_policy(workspace_id)
    except Exception as e:
        return "Failed", f"GET policy failed: {e}", "GetPolicyFailed"

    if "outbound" not in current_policy:
        current_policy["outbound"] = {}
    current_policy["outbound"]["publicAccessRules"] = {"defaultAction": outbound_action}

    url = f"{fabric_api_base}/workspaces/{workspace_id}/networking/communicationPolicy"
    headers = {"Authorization": f"Bearer {fabric_token}", "Content-Type": "application/json"}
    if etag:
        headers["If-Match"] = etag

    response = requests.put(url, headers=headers, json=current_policy)

    if response.status_code == 429:
        retry_after = int(response.headers.get("Retry-After", 60))
        print(f"  Rate limited. Waiting {retry_after}s...")
        time.sleep(retry_after)
        response = requests.put(url, headers=headers, json=current_policy)

    if response.status_code == 200:
        return "Success", outbound_action, None

    error_code = ""
    try:
        error_code = response.json().get("errorCode", "")
    except Exception:
        pass

    return "Failed", f"{response.status_code}: {response.text}", error_code


In [ ]:
# Item types that are compatible with AOP — never deleted
ALLOWED_ITEM_TYPES = {
    "Lakehouse", "Notebook", "SparkJobDefinition", "Environment",
    "Warehouse", "DataFlow", "DataPipeline", "CopyJob",
    "MirroredDatabase", "SQLEndpoint", "VariableLibrary",
}

# PBI item types that block AOP enablement, in required deletion order
PBI_DELETION_ORDER = [
    "Scorecard", "Exploration", "Dashboard", "App",
    "PaginatedReport", "Report", "SemanticModel",
]

# Maps item type → Fabric REST API endpoint segment (same as DeleteItems notebook)
# Types not in this map fall back to the generic "items" endpoint
ITEM_ENDPOINT_MAP = {
    "ApacheAirflowJob":   "ApacheAirflowJobs",
    "CopyJob":            "copyJobs",
    "DataFlow":           "dataflows",
    "DataPipeline":       "dataPipelines",
    "Environment":        "environments",
    "Eventhouse":         "eventhouses",
    "GraphQLApi":         "GraphQLApis",
    "KQLDashboard":       "kqlDashboards",
    "KQLQueryset":        "kqlQuerysets",
    "Lakehouse":          "lakehouses",
    "MirroredDatabase":   "mirroredDatabases",
    "Notebook":           "notebooks",
    "Reflex":             "reflexes",
    "SparkJobDefinition": "sparkJobDefinitions",
    "VariableLibrary":    "VariableLibraries",
    "Warehouse":          "warehouses",
    "Report":             "reports",
    "SemanticModel":      "semanticModels"
}


def get_workspace_items(workspace_id):
    """
    Lists all items in a workspace, handling pagination.
    API: GET /v1/workspaces/{workspaceId}/items
    """
    url = f"{fabric_api_base}/workspaces/{workspace_id}/items"
    headers = {"Authorization": f"Bearer {fabric_token}"}
    items = []
    while url:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
        items.extend(data.get("value", []))
        url = data.get("continuationUri")
    return items


def get_blocking_items(workspace_id):
    """Returns items that prevent AOP enablement (not in ALLOWED_ITEM_TYPES)."""
    all_items = get_workspace_items(workspace_id)
    return [item for item in all_items if item.get("type") not in ALLOWED_ITEM_TYPES]


def delete_item(workspace_id, item_id, item_type, max_retries=2):
    """
    Deletes a single Fabric item using the same endpoint mapping as DeleteItems notebook.
    API: DELETE /v1/workspaces/{workspaceId}/{endpoint}/{itemId}
    Returns True on success or 404 (already gone), False otherwise.
    """
    endpoint = ITEM_ENDPOINT_MAP.get(item_type, "items")
    url = f"{fabric_api_base}/workspaces/{workspace_id}/{endpoint}/{item_id}"
    headers = {"Authorization": f"Bearer {fabric_token}", "Content-Type": "application/json"}

    for _ in range(max_retries):
        response = requests.delete(url, headers=headers)
        if response.status_code in (200, 204):
            return True
        if response.status_code == 404:
            return True  # Already gone
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 60))
            print(f"    Rate limited. Waiting {retry_after}s...")
            time.sleep(retry_after)
            continue
        print(f"    Delete failed for {item_id}: {response.status_code} {response.text}")
        return False

    print(f"    Max retries exceeded for {item_id}.")
    return False


def delete_blocking_items(workspace_id, blocking_items):
    """
    Deletes blocking items in the correct order:
      1. PBI types in dependency order (Scorecard → … → SemanticModel)
      2. All remaining non-allowed types
    Returns a list of items that could not be deleted.
    """
    pbi_buckets = {kind: [] for kind in PBI_DELETION_ORDER}
    other_items = []

    for item in blocking_items:
        item_type = item.get("type", "")
        if item_type in pbi_buckets:
            pbi_buckets[item_type].append(item)
        else:
            other_items.append(item)

    failed = []

    for kind in PBI_DELETION_ORDER:
        for item in pbi_buckets[kind]:
            name = item.get("displayName", item["id"])
            print(f"    Deleting {kind} '{name}' ({item['id']})")
            if not delete_item(workspace_id, item["id"], kind):
                failed.append(item)

    for item in other_items:
        item_type = item.get("type", "unknown")
        name = item.get("displayName", item["id"])
        print(f"    Deleting {item_type} '{name}' ({item['id']})")
        if not delete_item(workspace_id, item["id"], item_type):
            failed.append(item)

    return failed


def clear_blocking_items_and_retry_aop(workspace_id, aop_must_be):
    """
    Resolves OutboundRestrictionNotEligible by deleting incompatible items,
    then retries set_workspace_aop.
    Returns same (status, detail, error_code) signature as set_workspace_aop.
    """
    print(f"  Resolving OutboundRestrictionNotEligible for {workspace_id}...")

    try:
        blocking_items = get_blocking_items(workspace_id)
    except Exception as e:
        return "Failed", f"Could not list workspace items: {e}", "GetItemsFailed"

    if not blocking_items:
        return "Failed", "No blocking items found but AOP still failed", "OutboundRestrictionNotEligible"

    print(f"  Found {len(blocking_items)} blocking item(s). Deleting...")
    failed_items = delete_blocking_items(workspace_id, blocking_items)

    if failed_items:
        names = ", ".join(f"{i.get('type')} '{i.get('displayName')}'" for i in failed_items)
        return "Failed", f"Could not delete: {names}", "ItemDeletionFailed"

    print("  All blocking items deleted. Retrying AOP change...")
    return set_workspace_aop(workspace_id, aop_must_be)


In [ ]:
results_schema = StructType([
    StructField("WorkspaceId",  StringType(),    True),
    StructField("ToChangeAOP",  BooleanType(),   True),
    StructField("AOP",          StringType(),    True),
    StructField("AOPMustBe",    StringType(),    True),
    StructField("CreationTime", TimestampType(), True),
    StructField("UserId",       StringType(),    True),
    StructField("AlertStatus",  StringType(),    True),
])

fix_results = []

if not workspaces:
    print("No workspaces to process.")
elif not fabric_token:
    print("Aborting: No Fabric token available.")
else:
    for row in workspaces:
        ws_id       = row["WorkspaceId"]
        aop_current = row["AOP"]
        aop_value   = row["AOPMustBe"]
        print(f"Setting AOP for {ws_id} → {aop_value}")
        api_call_time = datetime.now(timezone.utc)

        status, detail, error_code = set_workspace_aop(ws_id, aop_value)

        # If AOP cannot be enabled because incompatible items exist, remove them and retry
        if (status == "Failed"
                and error_code == "OutboundRestrictionNotEligible"
                and aop_value == "EnableWorkspaceOutboundAccessProtection"):
            status, detail, error_code = clear_blocking_items_and_retry_aop(ws_id, aop_value)

        print(f"  {status}: {detail}")

        if status == "Success":
            fix_results.append((
                ws_id,
                False,          # ToChangeAOP — fixed, no longer needs changing
                aop_value,      # AOP — new value equals AOPMustBe
                aop_value,      # AOPMustBe
                api_call_time,
                "system",
                "Fixed",
            ))
        else:
            alert_status = f"Error:{error_code}" if error_code else f"Error:{detail}"
            fix_results.append((
                ws_id,
                True,           # ToChangeAOP — still needs fixing
                aop_current,    # AOP — unchanged, original value from AOPAlertLogs
                aop_value,      # AOPMustBe
                api_call_time,
                "system",
                alert_status,
            ))

# Write fix results directly to AOPAlertLogs in Eventhouse
if fix_results:
    results_df = spark.createDataFrame(fix_results, schema=results_schema)
    try:
        results_df.write \
            .format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", kusto_cluster_uri) \
            .option("kustoDatabase", kusto_database) \
            .option("kustoTable", "AOPAlertLogs") \
            .option("accessToken", kusto_token) \
            .option("tableCreateOptions", "CreateIfNotExist") \
            .mode("Append") \
            .save()
        print(f"Written {len(fix_results)} fix result(s) to AOPAlertLogs.")
    except Exception as e:
        print(f"Failed to write fix results to Eventhouse: {e}")
else:
    print("No fix results to write back.")


In [ ]:
# --- Update watermark in Fix_AOP_Config ---
new_watermark_str = execution_start_time.strftime("%Y-%m-%dT%H:%M:%S")

try:
    spark.sql(f"""
        MERGE INTO {fix_aop_config_table} AS t
        USING (SELECT CAST('{new_watermark_str}' AS TIMESTAMP) AS LastRunTimestamp) AS s
        ON (1 = 1)
        WHEN MATCHED THEN
            UPDATE SET t.LastRunTimestamp = s.LastRunTimestamp, t.UpdatedAt = current_timestamp()
        WHEN NOT MATCHED THEN
            INSERT (LastRunTimestamp, UpdatedAt) VALUES (s.LastRunTimestamp, current_timestamp())
    """)
    print(f"Watermark updated to: {new_watermark_str}")
except Exception as e:
    print(f"Failed to update watermark: {e}")
